# AiSynth Labs · AI DRUG guide · Notebook 4
## Decide

**Author:** Toluwanimi Odunewu (@toluwanimiso)  
**Last updated:** 2026-10-30  
**Runtime:** about 5 to 8 minutes · **GPU needed:** no  
**Projects covered:** 15 to 20 (selectivity, off-target views, ranking, break the pipeline, transfer test, memo)  
**Licence:** CC BY 4.0

Part of the free "Get started in AI drug discovery" guide from AiSynth Labs. Run the cells from top to bottom. Change one thing at a time and write down what moved.

### Aim of this notebook
Turn everything into a decision: which other proteins to watch, a transparent ranking that survives a change of weights, a pipeline you have deliberately broken, a second target to test whether your pipeline travels, and a one-page memo.

**Runs on its own:** it rebuilds the dataset from Notebook 1 in the first cell.

### References
1. Zdrazil B et al. (2023). *The ChEMBL Database in 2023.* Nucleic Acids Res. [doi:10.1093/nar/gkad1004](https://doi.org/10.1093/nar/gkad1004).
2. Daina A, Michielin O, Zoete V (2019). *SwissTargetPrediction: updated data and new features for efficient prediction of protein targets of small molecules.* Nucleic Acids Res 47:W357-W364.
3. Pedregosa F et al. (2011). *Scikit-learn: machine learning in Python.* J Mach Learn Res 12:2825-2830.
4. Butina D (1999). J Chem Inf Comput Sci 39:747-750 [doi:10.1021/ci9803381](https://doi.org/10.1021/ci9803381).
5. RDKit: open-source cheminformatics, [rdkit.org](https://www.rdkit.org).

---
# Practical

In [ ]:
# --- SETUP: runs on Colab, Kaggle or on your own computer -------------------------------
import os, sys, random, importlib.util
ON_COLAB = 'google.colab' in sys.modules
ON_KAGGLE = os.path.exists('/kaggle/input')
if (ON_COLAB or ON_KAGGLE) and importlib.util.find_spec('rdkit') is None:
    print('Installing RDKit (about a minute) ...')
    !pip install -q rdkit
SEED = 42                      # fixed seed: you and everyone else should get the same numbers
random.seed(SEED)
import numpy as np
np.random.seed(SEED)
print(f'Colab: {ON_COLAB} | Kaggle: {ON_KAGGLE} | seed {SEED}')

In [ ]:
# --- IMPORTS, SETTINGS AND THE READY LIBRARY (the same steps as Notebook 1, projects 3 and 4) --------------
import json, time, urllib.parse, urllib.request
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import Descriptors, Lipinski, rdFingerprintGenerator
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams
from rdkit.ML.Cluster import Butina
RDLogger.DisableLog('rdApp.*')

# >>> Change these lines to work on another protein. CDK5 is CHEMBL4036, BACE1 is CHEMBL4822.
TARGET_NAME = 'ACHE'
TARGET_CHEMBL_ID = 'CHEMBL220'          # human acetylcholinesterase
ACTIVE_CUTOFF = 6.3                     # pIC50 at or above this counts as "active". It is a choice: say yours when you post.
PIC50_RANGE = (2, 12)                   # outside this range (IC50 below 1 picomolar or above 10 millimolar) a value is treated as an error and dropped
results = {}                            # every number you will want to post is saved here

CHEMBL = 'https://www.ebi.ac.uk/chembl/api/data/'

def get_json(url, tries=4):
    for k in range(tries):
        try:
            return json.load(urllib.request.urlopen(url, timeout=90))
        except Exception:
            time.sleep(2 + 3 * k)
    raise RuntimeError('could not reach ' + url)

def fetch_ic50(chembl_id):
    rows, offset = [], 0
    while True:
        q = urllib.parse.urlencode({'target_chembl_id': chembl_id, 'standard_type': 'IC50', 'standard_relation': '=',
                                    'assay_type': 'B', 'standard_units': 'nM', 'limit': 1000, 'offset': offset})
        page = get_json(CHEMBL + 'activity.json?' + q)
        rows += page['activities']
        if not page['page_meta']['next']:
            return pd.DataFrame(rows)
        offset += 1000

def rule_of_five_ok(m):
    ok = [Descriptors.ExactMolWt(m) <= 500, Lipinski.NumHAcceptors(m) <= 10, Lipinski.NumHDonors(m) <= 5, Descriptors.MolLogP(m) <= 5]
    return sum(ok) >= 3

def catalog(name):
    p = FilterCatalogParams(); p.AddCatalog(getattr(FilterCatalogParams.FilterCatalogs, name)); return FilterCatalog(p)
PAINS, BRENK = catalog('PAINS'), catalog('BRENK')

def build_library(chembl_id, apply_filters=True):
    """ChEMBL IC50 records -> one row per molecule (median pIC50) -> rule of five and PAINS filters."""
    raw = fetch_ic50(chembl_id)
    d = raw[['molecule_chembl_id', 'canonical_smiles', 'standard_value', 'document_year']].copy()
    d['standard_value'] = pd.to_numeric(d['standard_value'], errors='coerce')
    d = d.dropna(subset=['standard_value', 'canonical_smiles']); d = d[d['standard_value'] > 0]
    d['pIC50'] = 9 - np.log10(d['standard_value'])
    d = (d.groupby('molecule_chembl_id', as_index=False)
           .agg(smiles=('canonical_smiles', 'first'), pIC50=('pIC50', 'median'), document_year=('document_year', 'min')))
    d = d[d.pIC50.between(*PIC50_RANGE)]                  # drop the impossible potencies (see Notebook 1, project 3)
    d['mol'] = [Chem.MolFromSmiles(s) for s in d['smiles']]
    d = d[d['mol'].notna()].reset_index(drop=True)
    n_all = len(d)
    if apply_filters:
        d = d[[rule_of_five_ok(m) for m in d.mol]]
        d = d[[not PAINS.HasMatch(m) for m in d.mol]]
    return d.reset_index(drop=True), n_all

lib, n_all = build_library(TARGET_CHEMBL_ID)
print(f'{TARGET_NAME} ({TARGET_CHEMBL_ID}): {n_all:,} molecules in ChEMBL, {len(lib):,} left after the rule of five and PAINS')
results['library'] = dict(target=TARGET_NAME, molecules_all=n_all, molecules_library=len(lib))

## Project 15. Selectivity watch
A molecule that hits your target and ten others is a different story from one that hits only your target. ChEMBL records the other proteins each molecule was measured on. We list them for your 50 most potent molecules.

In [ ]:
# --- PROJECT 15: selectivity watch ----------------------------------------------------------------------------------
# For your most potent molecules, which OTHER proteins has ChEMBL seen them measured on? Those are the first counter-screens.
potent = lib.sort_values('pIC50', ascending=False).head(50)
ids = list(potent.molecule_chembl_id)

rows, offset = [], 0
while True:
    q = urllib.parse.urlencode({'molecule_chembl_id__in': ','.join(ids), 'standard_type': 'IC50', 'standard_relation': '=', 'assay_type': 'B',
                                'standard_units': 'nM', 'limit': 1000, 'offset': offset})
    page = get_json(CHEMBL + 'activity.json?' + q)
    rows += page['activities']
    if not page['page_meta']['next']: break
    offset += 1000
act = pd.DataFrame(rows)
act['standard_value'] = pd.to_numeric(act['standard_value'], errors='coerce')
act = act[(act.standard_value > 0) & (act.target_chembl_id != TARGET_CHEMBL_ID) & (act.target_organism == 'Homo sapiens')]
act['pIC50'] = 9 - np.log10(act.standard_value)
own = potent.set_index('molecule_chembl_id').pIC50
act['own_pIC50'] = act.molecule_chembl_id.map(own)
act['gap'] = act.own_pIC50 - act.pIC50                         # how much more potent on YOUR target (bigger = more selective)
off = (act.groupby(['target_chembl_id', 'target_pref_name'])
          .agg(molecules=('molecule_chembl_id', 'nunique'), median_pIC50=('pIC50', 'median'), median_gap=('gap', 'median'))
          .reset_index().sort_values('molecules', ascending=False).head(10).round(2))
print(f'{len(ids)} most potent {TARGET_NAME} molecules; the other human proteins they were also measured on (most molecules first):')
display(off)
results['p15'] = dict(molecules=len(ids), top_off_targets=off.head(5).to_dict('records'))

## Project 16. Two views of off-targets
SwissTargetPrediction predicts targets from structure alone. Paste its list and see where it agrees with what ChEMBL has measured. Agreement is a flag; disagreement is something to resolve before you trust either.

In [ ]:
# --- PROJECT 16: two views of off-targets ----------------------------------------------------------------------------
# Go to swisstargetprediction.ch, paste your shortlist (human), and copy the names of the predicted targets here:
stp_targets = []        # for example: ['Acetylcholinesterase', 'Butyrylcholinesterase', ...]

chembl_names = [n for n in off.target_pref_name]
def match(a, b):
    a, b = a.lower(), b.lower()
    return a in b or b in a
if not stp_targets:
    print('Paste the target names from SwissTargetPrediction into stp_targets above, then run this cell again.')
else:
    both = [n for n in chembl_names if any(match(n, s) for s in stp_targets)]
    only_chembl = [n for n in chembl_names if n not in both]
    only_stp = [s for s in stp_targets if not any(match(s, n) for n in chembl_names)]
    print('BOTH views (flag these):', both)
    print('only in ChEMBL measurements (resolve before trusting):', only_chembl)
    print('only in SwissTargetPrediction (resolve before trusting):', only_stp)
    results['p16'] = dict(both=both, only_chembl=only_chembl, only_stp=only_stp)

## Project 17. One transparent ranking
Combine five columns into one score with weights you write down, then shake the weights and see whether your top five survive. Held-out molecules only: the model never saw them.

In [ ]:
# --- PROJECT 17: one transparent ranking ---------------------------------------------------------------------------------
from sklearn.ensemble import RandomForestRegressor
from rdkit.Chem import Descriptors

gen2 = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
Xr = np.array([gen2.GetFingerprintAsNumPy(m) for m in lib.mol], dtype=np.uint8)
yr = lib.pIC50.values
rng = np.random.RandomState(SEED); idx = rng.permutation(len(lib)); n_tr = int(0.7 * len(lib))
tr, te = idx[:n_tr], idx[n_tr:]
rf = RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1).fit(Xr[tr], yr[tr])
trees = np.array([t.predict(Xr[te]) for t in rf.estimators_])

T = lib.iloc[te].copy().reset_index(drop=True)
T['predicted'], T['uncertainty'] = trees.mean(0), trees.std(0)                       # the spread across 200 trees
T['alerts'] = [len(BRENK.GetMatches(m)) for m in T.mol]
T['brain_like'] = [int(Descriptors.TPSA(m) <= 90 and Descriptors.MolWt(m) <= 450 and Descriptors.NumHDonors(m) <= 3) for m in T.mol]
f_don = gen2.GetFingerprint(Chem.MolFromSmiles('COC1=C(C=C2C(=C1)CC(C2=O)CC3CCN(CC3)CC4=CC=CC=C4)OC'))
T['sim_donepezil'] = DataStructs.BulkTanimotoSimilarity(f_don, [gen2.GetFingerprint(m) for m in T.mol])
T = T[T.sim_donepezil < 0.95].reset_index(drop=True)          # the known drug itself (and its salts) is not a new candidate
Z = lambda s: (s - s.mean()) / s.std()
cols = {'predicted': +1, 'uncertainty': -1, 'alerts': -1, 'brain_like': +1, 'sim_donepezil': +1}
W0 = {'predicted': 1.0, 'uncertainty': 0.5, 'alerts': 0.5, 'brain_like': 0.5, 'sim_donepezil': 0.25}          # YOUR weights: write them on the page
def score(w):
    return sum(w[c] * cols[c] * Z(T[c]) for c in cols)
T['score'] = score(W0)
base_top = T.sort_values('score', ascending=False).head(5)
display(base_top[['molecule_chembl_id', 'predicted', 'uncertainty', 'alerts', 'brain_like', 'sim_donepezil', 'score']].round(2))

# Sensitivity: change every weight by a random factor between 0.5 and 2, 500 times. How often does each pick stay in the top 5?
stay = {m: 0 for m in base_top.molecule_chembl_id}
for _ in range(500):
    w = {c: W0[c] * rng.uniform(0.5, 2.0) for c in W0}
    top5 = set(T.assign(s=score(w)).sort_values('s', ascending=False).head(5).molecule_chembl_id)
    for m in stay: stay[m] += m in top5
print('share of 500 weight changes in which the pick stays in the top 5:')
for m, c in stay.items(): print(f'  {m}: {c / 500:.0%}')
results['p17'] = dict(weights=W0, test_molecules=len(T), top5=base_top.molecule_chembl_id.tolist(), stays={m: round(c / 500, 2) for m, c in stay.items()},
                      top5_measured=[round(float(v), 2) for v in base_top.pIC50])

## Project 18. Run the pipeline, then break it
Change one setting at a time and watch what moves. The point is to learn which of your choices drives the result.

In [ ]:
# --- PROJECT 18: run the pipeline, then break it ---------------------------------------------------------------------------
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import roc_auc_score

def cluster_ids(mols, cutoff=0.2):
    g = rdFingerprintGenerator.GetRDKitFPGenerator(maxPath=5, fpSize=2048)
    fps = [g.GetFingerprint(m) for m in mols]
    d = []
    for i in range(1, len(fps)):
        d.extend([1 - s for s in DataStructs.BulkTanimotoSimilarity(fps[i], fps[:i])])
    cl = Butina.ClusterData(d, len(fps), cutoff, isDistData=True)
    grp = np.zeros(len(fps), dtype=int)
    for k, c in enumerate(cl): grp[list(c)] = k
    return grp, len(cl)

def run_pipeline(library, radius=3, cutoff=ACTIVE_CUTOFF, groups=None):
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=2048)
    X = np.array([gen.GetFingerprintAsNumPy(m) for m in library.mol], dtype=np.uint8)
    y = (library.pIC50.values >= cutoff).astype(int)
    grp, n_cl = groups if groups is not None else cluster_ids(library.mol)
    def auc(splits):
        a = []
        for tr, te in splits:
            if len(set(y[te])) < 2 or len(set(y[tr])) < 2: continue
            m = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1).fit(X[tr], y[tr]); a.append(roc_auc_score(y[te], m.predict_proba(X[te])[:, 1]))
        return float(np.mean(a)), float(np.std(a))
    r, c = auc(StratifiedKFold(3, shuffle=True, random_state=SEED).split(X, y)), auc(GroupKFold(3).split(X, y, groups=grp))
    return dict(molecules=len(y), active_share=round(float(y.mean()), 2), clusters=n_cl, auc_random=round(r[0], 3), spread_random=round(r[1], 3),
                auc_cluster=round(c[0], 3), spread_cluster=round(c[1], 3))

lib_all, _ = build_library(TARGET_CHEMBL_ID, apply_filters=False)         # the same molecules before the rule of five and PAINS
g_lib, g_all = cluster_ids(lib.mol), cluster_ids(lib_all.mol)
experiments = {'baseline': run_pipeline(lib, 3, ACTIVE_CUTOFF, g_lib),
               'fingerprint radius 2 instead of 3': run_pipeline(lib, 2, ACTIVE_CUTOFF, g_lib),
               'active cut-off 7.0 instead of ' + str(ACTIVE_CUTOFF): run_pipeline(lib, 3, 7.0, g_lib),
               'no filters (keep everything)': run_pipeline(lib_all, 3, ACTIVE_CUTOFF, g_all)}
table = pd.DataFrame(experiments).T
display(table)
results['p18'] = {k: v for k, v in experiments.items()}

## Project 19. The transfer test
Does a pipeline that looks good on one protein still look good on another with far less data?

In [ ]:
# --- PROJECT 19: the transfer test ----------------------------------------------------------------------------------------------
# The same pipeline on a second protein: CDK5 (a kinase, small dataset) against the one you started with.
other_id, other_name = ('CHEMBL4036', 'CDK5') if TARGET_CHEMBL_ID != 'CHEMBL4036' else ('CHEMBL220', 'ACHE')
lib2, n2 = build_library(other_id)
two = pd.DataFrame({TARGET_NAME: run_pipeline(lib, 3, ACTIVE_CUTOFF, g_lib), other_name: run_pipeline(lib2, 3, ACTIVE_CUTOFF)}).T
display(two)
results['p19'] = two.to_dict('index')
print('Read the SPREAD column too: with a few hundred molecules the folds disagree a lot, so a good-looking mean can be luck.')

## Project 20. The decision memo

In [ ]:
# --- PROJECT 20: the decision memo --------------------------------------------------------------------------------------------
# One page. Fill the blanks (...) in your own words, keep every number, and finish with a go or no-go.
p = results
memo = f"""
DECISION MEMO: {TARGET_NAME} ({TARGET_CHEMBL_ID}), {time.strftime('%d %b %Y')}

1. TARGET   Why this protein: ...
2. DATA     {p['library']['molecules_all']:,} molecules in ChEMBL, {p['library']['molecules_library']:,} after the rule of five and PAINS. Worrying values: ...
3. TRUST    Random-split AUC {p['p18']['baseline']['auc_random']} but cluster-split AUC {p['p18']['baseline']['auc_cluster']}. A prediction is believable when: ...
4. STRUCTURE  Chosen structure: ... Checklist residues: ...
5. OFF-TARGET RISK  Other proteins to counter-screen first: {', '.join(r['target_pref_name'] for r in p['p15']['top_off_targets'][:3])}
6. RANKED PICKS    {', '.join(p['p17']['top5'])} (weights written on the page; they stay in the top 5 in {round(100 * min(p['p17']['stays'].values()))}% to {round(100 * max(p['p17']['stays'].values()))}% of weight changes)
7. WEAKEST LINK    ...
8. WHAT WOULD CHANGE MY MIND   ...

DECISION: go / no-go on a first round of testing of ... compounds.
"""
print(memo)

---
# Your turn
**Task:** finish the memo in your own words and post it as one image with its limits line.

**Expected output:** a one-page memo with a number or figure under every heading, ending in go or no-go.

---
# Discussion
**What you learned:** a ranking is only as good as its weights, and a pipeline is only as general as its weakest dataset.

**What this cannot do, and where it breaks:** the off-target list only shows what ChEMBL happens to contain, not what is true. The ranking mixes a model prediction with simple rules and arbitrary weights. Two targets are an anecdote, not a rule. Nothing here is evidence that a compound works: it is a plan for what to test first.